In [ ]:
########Dataset directory
print('Starting Run')

Dataset_directories = '''Location where your data are stored. Images must be stored as single image planes with the following naming convention:
                            rXXcXXfXXpXX_cX
                            r= row (of the plate)
                            c= column (also of the plate)
                            f= field (identifies which field of view in the well this image corresopnds to)
                            p= plane (each confocal z stack will have many planes)
                            c= channel
                            '''

###Global package import
import os
import re
import glob
import signal
import pickle
import multiprocessing as mp
from multiprocessing import shared_memory
from itertools import product

import numpy as np
import pandas as pd
import scipy as sp
from scipy import stats, interpolate, ndimage
from scipy.ndimage import (
    gaussian_filter, gaussian_filter1d, label, find_objects, generate_binary_structure,
    sobel, binary_fill_holes, binary_closing
)
from scipy.stats import mode, sem, ttest_ind, mannwhitneyu
from scipy.interpolate import interp1d

import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
import pylab

import tifffile as tif
from tifffile import imread, imsave

import imagecodecs
import cv2
import PIL
from PIL import Image

import itk
import SimpleITK as sitk
from itk import BinaryThresholdImageFilter, MultiplyImageFilter
from skimage import io, color, measure, segmentation
from skimage.feature import peak_local_max
from skimage.segmentation import watershed
from skimage.transform import resize
from skimage.measure import marching_cubes, mesh_surface_area

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.ensemble import IsolationForest

import pyradiomics
from pyradiomics import radiomics
from radiomics import featureextractor

from statsmodels.stats.multitest import multipletests

from skimage.morphology import disk, closing, convex_hull_image

from joblib import Parallel, delayed

###Image processing: ImVols
class image_processing:
    def __init__(self, data_directory, redo):
        self.Images_dir=os.path.join(data_directory, 'Images/')
        self.ImVols_dir=os.path.join(data_directory, 'ImVols/')
        self.maxprojdir=os.path.join(data_directory, 'Max_Projections/')
        self.decon_dir =os.path.join(data_directory, 'ImVols_Deconvolved/')
        self.PSF_CF488A = tif.imread('/net/bmc-lab8/data/lab/boyer/users/jhday/Sanofi_Data/CF488A_PSF.tif')
        self.PSF_CF633 =  tif.imread('/net/bmc-lab8/data/lab/boyer/users/jhday/Sanofi_Data/CF633_PSF.tif')
        self.redo = redo
        
    def process_images(self):
        if not os.path.exists(self.ImVols_dir):
            os.makedirs(self.ImVols_dir)
            completed_ImVols = []
        else:
            # completed_ImVols = os.listdir(self.ImVols_dir)
            print('ImVols already generated')
            return
        # Regular expression pattern to match the filenames
        pattern = re.compile(r'r(\d{2})c(\d{2})f(\d{2,})p(\d{2,})-ch(\d)')
    
        # Dictionary to store the metadata for images based on rXXcXXfXX combination
        images_metadata = {}
        # Iterate through all files in the input directory
        for filename in os.listdir(self.Images_dir):
            match = pattern.match(filename)
            if match:
                r = match.group(1)
                c = match.group(2)
                f = match.group(3)
                p = int(match.group(4))  # plane
                ch = int(match.group(5))  # channel
    
                key = f"r{r}c{c}f{f}"

                            # === SKIP if already processed ===
                # if any(fname.startswith(key) and fname.endswith('.tiff') for fname in completed_ImVols):
                    # print(f'{key} already saved to ImVols.')
                    # continue

                if key not in images_metadata:
                    images_metadata[key] = {
                        'planes': set(),
                        'channels': set(),
                        'files': []
                    }
    
                images_metadata[key]['planes'].add(p)
                images_metadata[key]['channels'].add(ch)
                images_metadata[key]['files'].append((p, ch, filename))
    
        # Iterate through the metadata dictionary and process each combination incrementally
        for key, metadata in images_metadata.items():
            planes = sorted(metadata['planes'])
            channels = sorted(metadata['channels'])
            num_planes = len(planes)
            num_channels = len(channels)
    
            # Initialize the 3D arrays for each channel
            stacks = {ch: np.zeros((num_planes, 1080, 1080), dtype=np.uint16) for ch in channels}
    
            # Fill the 3D arrays with images
            for p, ch, filename in metadata['files']:
                img_path = os.path.join(self.Images_dir, filename)
                try:
                    tiff = tif.imread(img_path)
                    if tiff.shape != (1080, 1080):
                        print(f"File {filename} has unexpected shape {tiff.shape}, treating as zero array")
                        tiff = np.zeros((1080, 1080), dtype=np.uint16)
                except Exception as e:
                    print(f"Error reading file {filename}: {e}, treating as zero array")
                    tiff = np.zeros((1080, 1080), dtype=np.uint16)
    
                plane_index = planes.index(p)
                stacks[ch][plane_index, :, :] = tiff
    
            # Save each channel as a separate TIFF file
            for ch, stack in stacks.items():
                # Prepare ImageJ metadata
                ij_metadata = {
                    'axes': 'ZYX',
                    'Hyperstack': 'false',
                    'mode': 'composite',
                    'unit': 'micron',
                    'ImageJ': '1.52p',
                    'frames': 1,
                    'slices': num_planes,
                    'channels': 1
                }
                output_path = os.path.join(self.ImVols_dir, f"{key}_c{ch}.tiff")
                tif.imwrite(output_path, stack, metadata=ij_metadata)
    
    def max_projection(self, image_stack):
        # Assuming this is the provided function for maximum projection
        return np.max(image_stack, axis=0)
    
    def save_max_projections(self, channels=[1,2,3,4]):
        if not os.path.exists(self.maxprojdir):
            os.makedirs(self.maxprojdir)
        # else:
        #     print('Max projections already saved.')
        #     return
        maxproj_lst = os.listdir(self.maxprojdir)
        for channel in channels:
            # Iterate through all files in the input directory
            for filename in os.listdir(self.ImVols_dir):
                if filename in maxproj_lst and not self.redo:
                    print(f'{filename} already written to Max_Projections.')
                    continue
                if filename.endswith(f"_c{channel}.tiff"):
                    img_path = os.path.join(self.ImVols_dir, filename)
                    # Read the image stack
                    image_stack = tif.imread(img_path)
                    # Perform the maximum projection
                    max_proj = self.max_projection(image_stack)
                    # Create the output file path
                    output_path = os.path.join(self.maxprojdir, filename)
                    # Save the maximum projection image
                    tif.imwrite(output_path, max_proj, dtype=max_proj.dtype)

###Masking: Threshdata_c1/c2
class MicroscopySegmentation:
    def __init__(self, path=None, im_name=None, stain='hoechst', mask_channel='c1', data_channel='c2', watershed=False, mode_removal=False, double_mask=False, sphericity_thresh=0.7, mode='3d', exclude=False, redo=False):
        parameters = {
            'hoechst':       {'spacing':(5, 5, 1), 'sigma':5, 'lowerthresh':275, 'upperthresh':1000, 'minsize':300000, 'mindist':25, 'relvoxdim':(1,1,10), 'rz':2, 'rxy':10, 'absthresh':400},
            'lamp1'  :       {'spacing':(5, 5, 1), 'sigma':1, 'lowerthresh':750, 'upperthresh':10000, 'minsize':20, 'mindist':25, 'relvoxdim':(1,1,2), 'rz':2, 'rxy':5, 'absthresh':567},
            'hoechst_preEx': {'spacing':(5, 5, 1), 'sigma':2, 'lowerthresh':500, 'upperthresh':10000, 'minsize':10000, 'mindist':7, 'relvoxdim':(1,1,10), 'rz':1, 'rxy':2, 'absthresh':1000},
            'lamp1_preEx':   {'spacing':(5, 5, 1), 'sigma':2, 'lowerthresh':500, 'upperthresh':5000, 'minsize':15000, 'mindist':7, 'relvoxdim':(1,1,10), 'rz':1, 'rxy':2, 'absthresh':500},
            'ab'  :          {'spacing':(5, 5, 1), 'sigma':1, 'lowerthresh':500, 'upperthresh':100000, 'minsize':10, 'mindist':25, 'relvoxdim':(1,1,2), 'rz':1, 'rxy':4, 'absthresh':360},
            'ctnt':          {'spacing':(1,1,20), 'sigma':20, 'lowerthresh':300, 'upperthresh':5000, 'minsize':15000, 'mindist':7, 'relvoxdim':(1,1,10), 'rz':1, 'rxy':10, 'absthresh':1000},
            'ctnt_preEx':    {'spacing':(1,1,5), 'sigma':10, 'lowerthresh':300, 'upperthresh':5000, 'minsize':15000, 'mindist':7, 'relvoxdim':(1,1,10), 'rz':1, 'rxy':5, 'absthresh':1000},
            'CCC_mask':      {'spacing':(5,5,1), 'sigma':5, 'lowerthresh':225, 'upperthresh':10000, 'minsize':20, 'mindist':25, 'relvoxdim':(1,1,10), 'rz':1, 'rxy':20, 'absthresh':567},
            'CCC_mask_tight':{'spacing':(1,1,5), 'sigma':3, 'lowerthresh':300, 'upperthresh':10000, 'minsize':20, 'mindist':25, 'relvoxdim':(1,1,10), 'rz':1, 'rxy':10, 'absthresh':567},
            'CCC_mask_tight_pompe':{'spacing':(1,1,5), 'sigma':3, 'lowerthresh':350, 'upperthresh':10000, 'minsize':20, 'mindist':25, 'relvoxdim':(1,1,10), 'rz':1, 'rxy':10, 'absthresh':567},
            'gfp':           {'spacing':(1,1,5), 'sigma':10, 'lowerthresh':325, 'upperthresh':5000, 'minsize':15000, 'mindist':7, 'relvoxdim':(1,1,10), 'rz':1, 'rxy':20, 'absthresh':1000}
        }
        self.spacing=    parameters[stain]['spacing']
        self.sigma=      parameters[stain]['sigma']
        self.lowerthresh=parameters[stain]['lowerthresh']
        self.upperthresh=parameters[stain]['upperthresh']
        self.minsize=    parameters[stain]['minsize']
        self.mindist=    parameters[stain]['mindist']
        self.relvoxdim=  parameters[stain]['relvoxdim']
        self.absthresh=  parameters[stain]['absthresh']
        self.input_path= path
        self.image_name= im_name
        self.maskchan=   mask_channel
        self.datachan=   data_channel
        self.radiusz=    parameters[stain]['rz']
        self.radiusxy=   parameters[stain]['rxy']
        self.mode_removal= mode_removal
        self.double_mask = double_mask
        self.sphitythresh=sphericity_thresh
        self.mode=        mode
        self.watershed   =watershed
        self.exclude     =exclude
        self.stain_id    =stain
        self.redo        =redo

    def _exclude(self):
        """
        Process images in a directory, extract radiomics features, apply PCA using pre-loaded models,
        and exclude images based on their PC1 value.
        
        Parameters:
        - directory: The path to the directory containing image files.
        - pca_model_file: The path to the saved PCA model file.
        - scaler_file: The path to the saved scaler file.
        
        Returns:
        - exclude: A list of the first 9 characters of the filenames where PC1 < 0.
        """
        # Instantiate an empty list called exclude
        exclude = []
    
        # Load the PCA model and scaler
        with open('QC_pca_model.pkl', 'rb') as pca_file:
            pca_model = pickle.load(pca_file)
    
        with open('QC_minmax_scaler.pkl', 'rb') as scaler_file:
            scaler = pickle.load(scaler_file)
    
        # Generate a list of filenames in the directory
        filenames = [f for f in os.listdir(os.path.join(self.input_path,'Max_Projections')) if f.endswith(('.tif', '.tiff')) and f'_{hoechst_channel}' in f]
    
        # Instantiate the radiomics feature extractor (using default settings)
        extractor = featureextractor.RadiomicsFeatureExtractor()
        extractor.disableAllFeatures()
        extractor.enableFeatureClassByName('firstorder')
        extractor.enableFeatureClassByName('glrlm')       # Gray Level Run Length Matrix (GLRLM)
        extractor.enableFeatureClassByName('glszm')       # Gray Level Size Zone Matrix (GLSZM)
        extractor.enableFeatureClassByName('gldm')        # Gray Level Dependence Matrix (GLDM)
        extractor.enableFeatureClassByName('ngtdm')       # Neighboring Gray Tone Difference Matrix (NGTDM)
    
        # Iterate through each file and process
        count = 0
        for filename in filenames:
            count+=1
            print(count)
            # Load the image using tifffile
            image_path = os.path.join(os.path.join(self.input_path,'Max_Projections'), filename)
            image = tif.imread(image_path)
            # Create mask
            mask = image.copy()
            mask[mask > 0] = 1
            mask_sitk = sitk.GetImageFromArray(mask)
            image_sitk = sitk.GetImageFromArray(image)
            
            # Extract radiomics features
            try:
                features = extractor.execute(image_sitk, mask_sitk)
            except ValueError:
                exclude.append(filename[:9])  # Append the first 9 characters of the filename to exclude
                print(f'{filename} added to exclusion list from Value Error')
                continue
    
            # Convert features to a vector (excluding non-numeric or irrelevant entries)
            feature_vector = np.array([float(value) for key, value in features.items() if isinstance(value, (int, float, np.ndarray))])

            # Apply the pre-loaded scaler
            feature_vector_scaled = scaler.transform([feature_vector])
    
            # Apply the pre-loaded PCA model
            pca_result = pca_model.transform(feature_vector_scaled)
    
            # Check if the value for PC1 is less than 0
            if pca_result[0, 0] < -1:
                exclude.append(filename[:9])  # Append the first 9 characters of the filename to exclude
                print(f'{filename} added to exclusion list')
            else:
                print(f'{filename} is good')
            print(len(exclude))
        return exclude

    def remove_planes_based_on_mode(self,mode_threshold=250):
        #Goal: remove gossamer signal underneath nuclei
        image_3d = tif.imread(os.path.join(self.input_path,'ImVols',self.image_name))
        plane_modes = []
        # print(image_3d.shape)
        # Iterate over each z-plane and compute the mode
        for z in range(image_3d.shape[0]):
            plane_mode_value, _ = mode(image_3d[z, :, :], axis=None)#[0][0]
            plane_modes.append(plane_mode_value)
        # Convert the list of modes to a numpy array for easy manipulation
        plane_modes = np.array(plane_modes)
        # Find the index of the plane with the highest mode value
        max_mode_index = np.argmax(plane_modes)
        max_mode_value = plane_modes[max_mode_index]
        
        # Print the mode values and the max mode for debugging
        # print(f"Max mode value: {max_mode_value} at plane {max_mode_index}")
        if max_mode_index > 40 and max_mode_value > mode_threshold:
            print('Bad Image')
            return np.zeros_like(image_3d)
        
        if max_mode_value > mode_threshold:
            # Remove the plane with the highest mode value, all planes below it,
            # and two planes above it
            remove_planes = list(range(0, max_mode_index + 3))  # +3 to include the plane itself and two above it
            cleaned_image_3d = np.delete(image_3d, remove_planes, axis=0)
            print(f"Planes removed: {remove_planes}")
        else:
            cleaned_image_3d = image_3d
            print("No planes removed")
        
        return cleaned_image_3d

    def calculate_sphericity(self, image_3d):
        
        # Ensure the image is binary (1 for object, 0 for background)
        binary_image = (image_3d > 0).astype(np.uint8)
    
        # Compute the volume (number of voxels inside the object)
        volume = np.sum(binary_image)
    
        # Use marching cubes to extract a mesh for the object surface
        # Marching cubes works best with a smooth surface, so make sure the image is binary
        verts, faces, _, _ = marching_cubes(binary_image, level=0)
    
        # Calculate the surface area from the marching cubes output
        surface_area = mesh_surface_area(verts, faces)
    
        # Sphericity calculation
        sphericity = (np.pi ** (1/3) * (6 * volume) ** (2/3)) / surface_area
    
        return sphericity
        
    def absolute_thresh(self):
        image = tif.imread(os.path.join(self.input_path,'ImVols/', self.image_name))
        image[image < self.absthresh] = 0
        image[image >= self.absthresh] = 1
        return image.T

    def _set_zero_to_150(self):
        image_3d = tif.imread(os.path.join(self.input_path,'ImVols',self.image_name))
        image_3d[image_3d == 0] = 150
        return image_3d

    def _process_slice(self, mask2d_bool, se2d):
        print('slice')
        sl = binary_fill_holes(mask2d_bool)
        sl = binary_closing(sl,structure=se2d, border_value=1)
        return sl

    def postprocess_stack_joblib(self,
        mask3d,
        roll_radius = 50,
        n_jobs = -1,
        prefer = "threads"   # try "threads" first; if no speedup, try "processes"
        ):
        se = disk(roll_radius)
        print('Running Parallelized Image Processing Step.')
        # Run in parallel
        processed = Parallel(n_jobs=n_jobs, prefer=prefer, batch_size="auto")(
            delayed(self._process_slice)(mask3d[:,:,z], se) for z in range(mask3d.shape[2])
        )

        # Assemble output
        out = np.zeros_like(mask3d, dtype=bool)
        for z in range(len(processed)):
            out[:,:,z] = processed[z]

        return out
    
    def adaptive_thresh(self):
        # Read the image
        ImageType = itk.Image[itk.F, 3]  # Float type, 3D image
        if self.mode_removal:
            image_array = self.remove_planes_based_on_mode()
            image_array = image_array.astype(np.float32)
            image = itk.GetImageFromArray(image_array)

        elif self.double_mask:
            image_array = self._set_zero_to_150()
            image_array = image_array.astype(np.float32)
            image = itk.GetImageFromArray(image_array)
        else:
            reader = itk.ImageFileReader[ImageType].New()
            reader.SetFileName(os.path.join(self.input_path,'ImVols/', self.image_name))
            reader.Update()
            image = reader.GetOutput()
            # image_array = tif.imread(os.path.join(self.input_path,'ImVols',self.image_name))
            # image_array[image_array > self.upperthresh] = 0
            # image = itk.GetImageFromArray(image_array)


        image.SetSpacing(self.spacing)
        smoothing_filter = itk.SmoothingRecursiveGaussianImageFilter.New(Input=image,sigma=self.sigma)
        smoothing_filter.Update()
        image = smoothing_filter.GetOutput()
        
        # Apply a local statistic filter: here, we use a mean filter as a base for adaptive thresholding
        mean_filter = itk.MeanImageFilter[ImageType, ImageType].New()
        mean_filter.SetInput(image)

        # Define the radius of the neighborhood with non-uniform values
        radius = itk.Size[3]()  # Create a size object for a 3D image
        radius[0] = self.radiusxy  # Size along z
        radius[1] = self.radiusxy  # Size along y, 5 for Lamp1
        radius[2] = self.radiusz  # Size along x, 5 for Lamp1

        mean_filter.SetRadius(radius)  # Set the custom, non-uniform radius
        mean_filter.Update()  # Apply the filter
        
        filtered_image = mean_filter.GetOutput()

        # For actual thresholding, we use BinaryThresholdImageFilter
        threshold_filter = itk.BinaryThresholdImageFilter[ImageType, ImageType].New()
        threshold_filter.SetInput(filtered_image)
        threshold_filter.SetLowerThreshold(self.lowerthresh)  # Lamp1:250
        threshold_filter.SetUpperThreshold(self.upperthresh) # Lamp1:1000
        threshold_filter.SetInsideValue(1)  # Pixel value for pixels within the threshold range
        threshold_filter.SetOutsideValue(0)  # Pixel value for pixels outside the threshold range
        threshold_filter.Update()
        imgthresh = threshold_filter.GetOutput()

        # Transpose the image for visualization
        PermuteFilterType = itk.PermuteAxesImageFilter[ImageType]
        permute_filter = PermuteFilterType.New()
        permute_filter.SetInput(imgthresh)
        permute_filter.SetOrder([2, 1, 0])
        permute_filter.Update()
        
        # Write the output to a numpy array
        itk_image = permute_filter.GetOutput()
        image_array = itk.GetArrayFromImage(itk_image).astype(np.uint8)
        print(image_array.shape)
        image_array_out = self.postprocess_stack_joblib(image_array)
        # # Perform plane-wise binary hole filling (for each z-plane)
        # se = disk(50)
        # count = 0
        # for z in range(image_array.shape[2]):  # Iterate over each z-plane (x-y slices)
        #     count+=1
        #     print(f'Starting plane {count}')
        #     slice_im = image_array[:, :, z]
        #     if not slice_im.any():
        #         image_array[:, :, z] = slice_im
        #         continue
        #     slice_im = binary_fill_holes(slice_im)  # Fill holes in each 2D slice
        #     slice_im = closing(slice_im, se)
        #     image_array[:, :, z] = slice_im
        return image_array_out

    def label_and_trim(self, mask):
        # Find connected components in 3D
        labels, num_labels = ndi.label(mask, structure=ndi.generate_binary_structure(3, 1))
        label_sizes = np.bincount(labels.flatten())
        print(f'{num_labels} Unique regions identified')
        mask_remove_small = labels.copy()
        for label, size in enumerate(label_sizes):
            if size < self.minsize:
                mask_remove_small[labels == label] = 0
        num_labels_truncated = np.unique(mask_remove_small)
        print(f'{len(label_sizes) - len(num_labels_truncated)} small regions removed')
        return mask_remove_small, num_labels_truncated


    def find_bounding_box_3d(self, region):
        x, y, z = np.where(region)
        min_x, max_x = np.min(x), np.max(x)
        min_y, max_y = np.min(y), np.max(y)
        min_z, max_z = np.min(z), np.max(z)
        return min_x, min_y, min_z, max_x, max_y, max_z

    def getmaskedimage(self, mode='adaptive'):
        source_imagepath = os.path.join(self.input_path,'ImVols', self.image_name).replace(self.maskchan, self.datachan)
        source_image     = tif.imread(source_imagepath)
        if mode == 'adaptive':
            image_mask       = self.adaptive_thresh()
        elif mode == 'absolute':
            image_mask       = self.absolute_thresh()
        masked_image     = image_mask.T.astype(np.uint16) * source_image #(1-image_mask.T).astype(np.uint16)

        return masked_image

    def getregions_3d(self):
        source_imagepath = os.path.join(self.input_path,'ImVols', self.image_name).replace(self.maskchan, self.datachan)
        source_image     = tif.imread(source_imagepath)
        if self.watershed:
            labeled_image=self.watershed_segment()
        else:
            masked_image=self.adaptive_thresh()
            labeled_image, num_labels = self.label_and_trim(masked_image)
            labeled_image=labeled_image.T
        # print(source_image.shape)
        # print(labeled_image.shape)
        regions = []
        unique_labels = np.unique(labeled_image)
        # print(f'{len(unique_labels)} unique labels detected')
        # print(unique_labels)
        for label in unique_labels:
            if label == 0:  # Skip the background
                continue
            #Isolate the object deliniated by the label for a given loop
            region_mask = (labeled_image == label)
            region_mask[region_mask>0] = 1
            # Calculate the size of the labeled region
            region_size = np.sum(region_mask)
            # Skip regions smaller than self.minsize
            # print(region_size)
            if region_size < self.minsize:
                continue
            touches_y_border = np.any(region_mask[:, 0, :]) or np.any(region_mask[:, -1, :])
            touches_x_border = np.any(region_mask[:, :, 0]) or np.any(region_mask[:, :, -1])
            if touches_y_border or touches_x_border:
                # print(f"Label {label} touches the border and will be skipped")
                continue
            #Crop the image to only contain our object in it's minimal bounding box
            min_x, min_y, min_z, max_x, max_y, max_z = self.find_bounding_box_3d(region_mask)
            cropped_region = source_image[min_x:max_x+1, min_y:max_y+1, min_z:max_z+1] * region_mask[min_x:max_x+1, min_y:max_y+1, min_z:max_z+1]
            try:
                sphity = self.calculate_sphericity(cropped_region)
            except ValueError:
                continue
            # print(sphity)
            if sphity > self.sphitythresh:
                regions.append(cropped_region)
                
        return regions

    def process_dataset_threshold(self, mode='adaptive'):
        print('Starting threshold program...')
        # Get the list of source image files
        source_ims_full = [x for x in os.listdir(os.path.join(self.input_path, 'ImVols')) if f'_{self.maskchan}' in x]
        # Exclude files based on exclusion list if applicable
        if self.exclude:
            print('exclude')
            excludelst = self._exclude()
            source_ims = [filename for filename in source_ims_full if not any(filename.startswith(prefix[:9]) for prefix in excludelst)]
        else:
            source_ims = source_ims_full

        print(f'Source Ims contains {len(source_ims)} images...')
    
        # Directory for saving processed files
        savedir = os.path.join(self.input_path, f'ThreshData_{self.datachan}_/')#{self.stain_id} Saving the directory with the stain_id was awkward and I just delete it manually anyway
        if not os.path.exists(savedir):
            os.makedirs(savedir)
        for im in source_ims:
            self.image_name = im
    
            # Determine file path for saving the output
            output_path = os.path.join(savedir, f'{im[:-5]}.tiff')
    
            # Check if the file has already been processed
            if os.path.exists(output_path) and not self.redo:
                print(f'{im[:-5]} Already processed.')
                continue
    
            # If not processed, perform the computationally intensive parts
            image_hoechst = re.sub(r'_c\d+', f'_{hoechst_channel}', im)
            hoechst = tif.imread(os.path.join(self.input_path, 'ImVols', image_hoechst))
    
            # Apply the mask
            masked = self.getmaskedimage(mode)
    
            # Save the processed image
            if self.double_mask:
                tif.imwrite(os.path.join(self.input_path, 'ImVols', f'{im[:-5]}{self.datachan}.tiff'), masked)
            else:
                masked[masked != 0] = 1                       ###In masking, sets all non-zero voxels to 1, giving the typical output for saving in Threshdata folders.
                tif.imwrite(output_path, masked)
    
            print(im)

###Radiomics QC: inliers/outliers lists and qc data

class Radiomics_QC:
    def __init__(self, data_dir,c1,c2,redo=False):
        self.datadir = data_dir
        self.c1 = c1
        self.c2 = c2
        self.redo=redo
        self.digit_arrays = {
            '0': np.array([[0,1,1,1,0],
                           [1,0,0,0,1],
                           [1,0,0,0,1],
                           [1,0,0,0,1],
                           [1,0,0,0,1],
                           [1,0,0,0,1],
                           [0,1,1,1,0]], dtype=bool),
            '1': np.array([[0,0,1,0,0],
                           [0,1,1,0,0],
                           [1,0,1,0,0],
                           [0,0,1,0,0],
                           [0,0,1,0,0],
                           [0,0,1,0,0],
                           [1,1,1,1,1]], dtype=bool),
            '2': np.array([[0,1,1,1,0],
                           [1,0,0,0,1],
                           [0,0,0,0,1],
                           [0,0,0,1,0],
                           [0,0,1,0,0],
                           [0,1,0,0,0],
                           [1,1,1,1,1]], dtype=bool),
            '3': np.array([[0,1,1,1,0],
                           [1,0,0,0,1],
                           [0,0,0,0,1],
                           [0,0,1,1,0],
                           [0,0,0,0,1],
                           [1,0,0,0,1],
                           [0,1,1,1,0]], dtype=bool),
            '4': np.array([[0,0,0,1,0],
                           [0,0,1,1,0],
                           [0,1,0,1,0],
                           [1,0,0,1,0],
                           [1,1,1,1,1],
                           [0,0,0,1,0],
                           [0,0,0,1,0]], dtype=bool),
            '5': np.array([[1,1,1,1,1],
                           [1,0,0,0,0],
                           [1,1,1,1,0],
                           [0,0,0,0,1],
                           [0,0,0,0,1],
                           [1,0,0,0,1],
                           [0,1,1,1,0]], dtype=bool),
            '6': np.array([[0,0,1,1,0],
                           [0,1,0,0,0],
                           [1,0,0,0,0],
                           [1,1,1,1,0],
                           [1,0,0,0,1],
                           [1,0,0,0,1],
                           [0,1,1,1,0]], dtype=bool),
            '7': np.array([[1,1,1,1,1],
                           [0,0,0,0,1],
                           [0,0,0,1,0],
                           [0,0,1,0,0],
                           [0,1,0,0,0],
                           [0,1,0,0,0],
                           [0,1,0,0,0]], dtype=bool),
            '8': np.array([[0,1,1,1,0],
                           [1,0,0,0,1],
                           [1,0,0,0,1],
                           [0,1,1,1,0],
                           [1,0,0,0,1],
                           [1,0,0,0,1],
                           [0,1,1,1,0]], dtype=bool),
            '9': np.array([[0,1,1,1,0],
                           [1,0,0,0,1],
                           [1,0,0,0,1],
                           [0,1,1,1,1],
                           [0,0,0,0,1],
                           [0,0,0,1,0],
                           [0,1,1,0,0]], dtype=bool),
            '_': np.array([[0,0,0,0,0],
                           [0,0,0,0,0],
                           [0,0,0,0,0],
                           [0,0,0,0,0],
                           [0,0,0,0,0],
                           [0,0,0,0,0],
                           [1,1,1,1,1]], dtype=bool)
            }
    def generate_qc_data(self, channel):
        # Initialize the extractor with the .yaml configuration
        extractor = featureextractor.RadiomicsFeatureExtractor()
        
        # Disable GLCM by setting the feature classes manually
        extractor.disableAllFeatures()
        extractor.enableFeatureClassByName('firstorder')
        # extractor.enableFeatureClassByName('shape')
        # extractor.enableFeatureClassByName('glcm')        # Gray Level Co-occurrence Matrix (GLCM)
        extractor.enableFeatureClassByName('glrlm')       # Gray Level Run Length Matrix (GLRLM)
        extractor.enableFeatureClassByName('glszm')       # Gray Level Size Zone Matrix (GLSZM)
        extractor.enableFeatureClassByName('gldm')        # Gray Level Dependence Matrix (GLDM)
        extractor.enableFeatureClassByName('ngtdm')       # Neighboring Gray Tone Difference Matrix (NGTDM)
        
        imdir = os.path.join(self.datadir, 'Max_Projections/')
        mskdir = os.path.join(self.datadir, f'ThreshData_{channel}_/')
        
        count=0
        datafile_name = f'qc_data_{channel}.csv'
        dataframe_file = os.path.join(self.datadir, datafile_name)
        header_written = os.path.exists(dataframe_file)########################################
        if self.redo and header_written:
            os.remove(dataframe_file) #wipe the dataframe file if the operation is being redone
            header_written = False

        for filename in [x for x in os.listdir(imdir) if f'_{channel}' in x]:
            count += 1
            print(f'Processing file {count}: {filename}')
            # Add filename as the first column            
            if os.path.exists(dataframe_file):
                existing_df = pd.read_csv(dataframe_file)
                if filename in existing_df['Filename'].values:
                    print(f'Skipping {filename}, already processed.')
                    continue
        
            # Read image
            image = tif.imread(os.path.join(imdir, filename))
            # image = np.max(image, axis=0)  # Max projection (adjust if needed)
            image_sitk = sitk.GetImageFromArray(image)
            
            # Create mask
            try:
                msk = tif.imread(os.path.join(mskdir,filename))
            except FileNotFoundError:
                print(f'{filename} represented in MaxProjections but not Threshdata, continuing...')
                continue
            # msk_vox_count = np.count_nonzero(msk)
            msk = np.max(msk, axis=0)
            mask_sitk = sitk.GetImageFromArray(msk)
            
            # Extract radiomics features
            try:
                result = extractor.execute(image_sitk, mask_sitk)
            except ValueError:
                print(f'Bad image, skipping {filename}')
                continue
            # Convert the result (a dictionary) to a pandas Series and add metadata
            result_series = pd.Series(result)
            result_series['Filename'] = filename
            # result_series['Vox_Count'] = msk_vox_count
    
            # Append the result to the CSV file
            result_df = pd.DataFrame([result_series])
            
            result_df.to_csv(dataframe_file, mode='a', header=not header_written, index=False)
            header_written = True  # Only write header once

    def generate_qc_scores(self, channel, thresh):
        # Load the data
        datafile_name = f'qc_data_{channel}.csv'
        dataframe_file = os.path.join(self.datadir, datafile_name)
        df = pd.read_csv(dataframe_file)
        df = df.fillna(0)
    
        # Select only numerical columns for normalization and PCA
        numerical_df = df.select_dtypes(include='number')
        if 'Anomaly_Score' in numerical_df.columns:
            numerical_df = numerical_df.drop(columns=['Anomaly_Score'])
            
        # Step 1: Train Isolation Forest with auto contamination
        iso_forest = IsolationForest(n_estimators=500, contamination='auto', random_state=42)
        iso_forest.fit(numerical_df)
        
        # Step 2: Get anomaly scores and calculate threshold
        scores = iso_forest.decision_function(numerical_df)
        threshold = np.percentile(scores, thresh)
        
        # Add scores and classification to the DataFrame
        df['Anomaly_Score'] = scores
        df['Inlier/Outlier'] = ['Outlier' if score < threshold else 'Inlier' for score in scores]
    
        # Save the updated DataFrame back to CSV (optional)
        updated_file = dataframe_file
        df.to_csv(dataframe_file, index=False)
        print(f"Updated DataFrame saved to {datafile_name}")
    
    def plot_qc_data(self):
        # Load the data
        dataframe_file_1 = os.path.join(self.datadir, 'qc_data_c1.csv')
        dataframe_file_2 = os.path.join(self.datadir, 'qc_data_c4.csv')
        df1 = pd.read_csv(dataframe_file_1)
        df2 = pd.read_csv(dataframe_file_2)
    
        # Ensure both dataframes have necessary columns
        required_columns = ['Inlier/Outlier']
        if not all(col in df1.columns for col in required_columns) or not all(col in df2.columns for col in required_columns):
            raise ValueError("Both dataframes must contain 'Inlier/Outlier' column.")
    
        # Identify shared outliers (outliers in both datasets)
        shared_outliers = set(df1[df1['Inlier/Outlier'] == 'Outlier'].index).intersection(
            df2[df2['Inlier/Outlier'] == 'Outlier'].index
        )
        shared_outlier_proportion = len(shared_outliers) / min(len(df1), len(df2))
    
        print(f"Proportion of shared outliers: {shared_outlier_proportion:.2%}")
    
        # Prepare the plots
        fig, axes = plt.subplots(1, 2, figsize=(16, 6))
        for i, (df, title, ax) in enumerate(zip([df1, df2], ['Dataset 1', 'Dataset 2'], axes)):
            # Select only numerical columns for normalization and PCA
            numerical_df = df.select_dtypes(include='number')
            
            # Min-Max normalize each column (between 0 and 1)
            scaler = MinMaxScaler()
            normalized_data = scaler.fit_transform(numerical_df)
            
            # Perform PCA on normalized data
            n_components = 2
            pca = PCA(n_components=n_components)
            pca_transformed = pca.fit_transform(normalized_data)
            
            # Create a DataFrame for the PCA-transformed data
            pca_df = pd.DataFrame(pca_transformed, columns=[f'PC{i+1}' for i in range(n_components)])
            pca_df['Inlier/Outlier'] = df['Inlier/Outlier']
            pca_df['IsSharedOutlier'] = pca_df.index.isin(shared_outliers)
    
            # Calculate explained variance ratio for labeling
            explained_variance = pca.explained_variance_ratio_
    
            # Plot PCA with distinct color coding for shared outliers
            for label, color in zip(['Inlier', 'Outlier', 'Shared Outlier'], ['blue', 'red', 'purple']):
                if label == 'Shared Outlier':
                    subset = pca_df[pca_df['IsSharedOutlier']]
                else:
                    subset = pca_df[(pca_df['Inlier/Outlier'] == label) & (~pca_df['IsSharedOutlier'])]
                ax.scatter(subset['PC1'], subset['PC2'], label=label, alpha=0.7, c=color)
            
            ax.set_title(f'PCA Plot for {title}')
            ax.set_xlabel(f'PC1 ({explained_variance[0]*100:.2f}%)')
            ax.set_ylabel(f'PC2 ({explained_variance[1]*100:.2f}%)')
            ax.legend()
            ax.grid(True)
        
        # Adjust layout and show the plots
        plt.tight_layout()
        plt.show()

    def tile_images(self, channel, image_ext=".tiff"):
        """
        Reads images, tiles inlier and outlier images separately, and saves two large TIFF files.
    
        Parameters:
            channel (str): Channel identifier to filter images.
            image_ext (str): Image file extension (default: ".tiff").
        """
        # Load QC data
        qc_data_path = os.path.join(self.datadir, f'qc_data_{channel}.csv')
        qc_data = pd.read_csv(qc_data_path)
    
        # Ensure QC data contains necessary columns
        if 'Filename' not in qc_data.columns or 'Inlier/Outlier' not in qc_data.columns:
            raise ValueError("QC data must contain 'Filename' and 'Inlier/Outlier' columns.")
        
        # Separate inliers and outliers
        inlier_files = set(qc_data[qc_data['Inlier/Outlier'] == 'Inlier']['Filename'])
        outlier_files = set(qc_data[qc_data['Inlier/Outlier'] == 'Outlier']['Filename'])
    
        # Regex to match images for the specified channel
        pattern = re.compile(rf"(r\d{{2}}c\d{{2}}f\d{{2}}_{channel})")
        
        # Group images by inliers and outliers
        inlier_images = []
        outlier_images = []
        for file in os.listdir(os.path.join(self.datadir, 'Max_Projections/')):
            if file.endswith(image_ext) and f'_{channel}' in file:
                match = pattern.search(file)
                if match:
                    img_path = os.path.join(self.datadir, 'Max_Projections/', file)
                    if file in inlier_files:
                        inlier_images.append(img_path)
                    elif file in outlier_files:
                        outlier_images.append(img_path)

        # Helper function to create a tiled image
        def create_tiled_image(image_paths, output_path):
            if not image_paths:
                print(f"No images found for {output_path}. Skipping.")
                return
            
            overlay_images = []
            for file in image_paths:
                img = tif.imread(file)
                img_normalized = (img / img.max() * 65535).astype(np.uint16)
                imageID = self._digitseq(file)
                textmap = self._string_to_bitmap(imageID)
                img_IDd= self._overlay_bitmap_on_image(img_normalized,textmap)
                overlay_images.append(img_IDd)
            
            # Tile dimensions
            img_height, img_width = overlay_images[0].shape
            columns = int(np.ceil(np.sqrt(len(overlay_images))))
            rows = int(np.ceil(len(overlay_images) / columns))
            
            # Create the massive tiled canvas
            tiled_image = np.zeros((rows * img_height, columns * img_width), dtype=np.uint16)
            
            # Add images to the canvas
            for index, img in enumerate(overlay_images):
                x = (index % columns) * img_width
                y = (index // columns) * img_height
                tiled_image[y:y + img_height, x:x + img_width] = img
            
            # Save the final tiled image
            tif.imwrite(output_path, tiled_image)
            print(f"Tiled image saved as {output_path}")
    
        # Create and save tiled images for inliers and outliers
        inlier_output_path = os.path.join(self.datadir, f'Tile_{channel}_inliers.tiff')
        outlier_output_path = os.path.join(self.datadir, f'Tile_{channel}_outliers.tiff')
        create_tiled_image(inlier_images, inlier_output_path)
        create_tiled_image(outlier_images, outlier_output_path)

    def _digitseq(self, impath):
        # rXXcXXfXX_cX.tiff
        row  = impath[-16:-14]
        col  = impath[-13:-11]
        field= impath[-10:-8]
        print(row+col+field)
        return row+col+field

    def _string_to_bitmap(self, text):
        """Convert a string of digits/underscores to a combined boolean array."""
        char_width = 5
        spacing = 1
        height = 7
        total_width = len(text) * (char_width + spacing) - spacing
    
        bitmap = np.zeros((height, total_width), dtype=bool)
        
        for i, char in enumerate(text):
            if char in self.digit_arrays:
                digit = self.digit_arrays[char]
            else:
                digit = self.digit_arrays['_']
            start_x = i * (char_width + spacing)
            bitmap[:, start_x:start_x + char_width] = digit
        
        return bitmap

    def _overlay_bitmap_on_image(self, image, bitmap, x_offset=0, y_offset=0):
        """Overlay a bitmap onto a 16-bit image at the specified offset."""
        h, w = bitmap.shape
        image[y_offset:y_offset+h, x_offset:x_offset+w] = np.where(
            bitmap, 65535, 0
        )
        return image
                

    def save_shared_inliers(self, output_file='shared_inliers.txt'):
        """
        Saves a list of filenames representing shared inliers to a text file.
        """
        
        # Load the QC data for the two datasets
        dataframe_file_1 = os.path.join(self.datadir, f'qc_data_{self.c1}.csv')
        dataframe_file_2 = os.path.join(self.datadir, f'qc_data_{self.c2}.csv')
        df1 = pd.read_csv(dataframe_file_1)
        df2 = pd.read_csv(dataframe_file_2)
        
        # Ensure the required columns are present
        required_columns = ['Filename', 'Inlier/Outlier']
        if not all(col in df1.columns for col in required_columns) or not all(col in df2.columns for col in required_columns):
            raise ValueError("Both dataframes must contain 'Filename' and 'Inlier/Outlier' columns.")
            
        def normalize_filename(filename):
            return filename.rsplit('_', 1)[0]  # Split on the last underscore and remove channel-specific suffix
            
        # Create sets of normalized filenames for inliers in both datasets
        inliers1 = set(df1[df1['Inlier/Outlier'] == 'Inlier']['Filename'].map(normalize_filename).str.strip().str.lower())
        inliers2 = set(df2[df2['Inlier/Outlier'] == 'Inlier']['Filename'].map(normalize_filename).str.strip().str.lower())
        shared_inliers = inliers1.intersection(inliers2)
        # Save shared inliers to a text file
        output_path = os.path.join(self.datadir, output_file)
        with open(output_path, 'w') as f:
            for filename in shared_inliers:
                f.write(filename + '\n')
    
        print(f"Shared inliers saved to {output_path}")
    def workup(self):
        for c in [self.c1,self.c2]:
            self.generate_qc_data(c)
            print('done')
            self.generate_qc_scores(c, 20)
            self.tile_images(c)
        # self.plot_qc_data()
        self.save_shared_inliers()


###Acting Commands
gfp_channel     = 'c3'
ctnt_channel    = 'c2'
Lamp1_channel   = 'c4'
hoechst_channel = 'c1'
re_do = True
Pre = True
for datadir in Dataset_directories:
    # print(f'Starting program on {datadir[20:]}')
    ip=image_processing(datadir,redo=re_do)
    ip.process_images()
    ip.save_max_projections()
    if not Pre:
        segmentation = MicroscopySegmentation(path=datadir, stain='hoechst', mask_channel=hoechst_channel, data_channel=hoechst_channel, double_mask=False, mode='3d', sphericity_thresh = 0.5, exclude=False, watershed=False, redo=re_do)
        segmentation.process_dataset_threshold(mode='absolute')
        segmentation = MicroscopySegmentation(path=datadir, stain='ctnt', mask_channel=ctnt_channel, data_channel=Lamp1_channel, double_mask=True, mode='3d', sphericity_thresh = 0.5, exclude=False, watershed=False, redo=re_do)
        segmentation.process_dataset_threshold()
    else:
        segmentation = MicroscopySegmentation(path=datadir, stain='ctnt_preEx', mask_channel=ctnt_channel, data_channel=ctnt_channel, double_mask=False, mode='3d', sphericity_thresh = 0.5, exclude=False, watershed=False, redo=re_do)
        segmentation.process_dataset_threshold(mode='absolute')
        segmentation = MicroscopySegmentation(path=datadir, stain='hoechst_preEx', mask_channel=hoechst_channel, data_channel=hoechst_channel, double_mask=False, mode='3d', sphericity_thresh = 0.5, exclude=False, watershed=False, redo=re_do)
        segmentation.process_dataset_threshold(mode='absolute')

    qc = Radiomics_QC(datadir,hoechst_channel,ctnt_channel, redo=re_do)
    qc.workup()
    print('Data Processing Completed.')